# Module 5 — Data Preparation & Feature Engineering
## Hands-On Tutorial for Computational Materials Science

**Level:** IIT M.Tech / PhD Applied Materials / Computational Materials  
**Duration:** One 3-hour tutorial + independent exercises  
**Prerequisites:** Modules 1–4

---

## Central idea

Machine learning does not begin with choosing an algorithm.

For a materials-science problem, the workflow is:

\[
\boxed{
\text{Materials data}
\rightarrow
\text{cleaning}
\rightarrow
\text{quality control}
\rightarrow
\text{representation}
\rightarrow
\text{feature engineering}
\rightarrow
\text{dimensionality reduction}
\rightarrow
\text{train/test split}
\rightarrow
\text{ML}
}
\]

The quality of the representation and data preparation often matters as much as the choice of machine-learning algorithm.

### Learning objectives

By the end of this tutorial, students should be able to:

- Inspect and clean a materials dataset.
- Identify numerical and categorical variables.
- Handle missing values appropriately.
- Detect possible outliers.
- Understand scaling and normalization mathematically.
- Encode categorical variables.
- Engineer physically meaningful materials descriptors.
- Understand feature selection and data leakage.
- Perform a reproducible train/test split.
- Understand why preprocessing must be learned from training data.
- Apply PCA and interpret principal components.
- Visualize high-dimensional materials data.
- Build a preprocessing pipeline using scikit-learn.
- Prepare a dataset correctly for subsequent ML models.


# 1. Scientific Python stack

We will use:

- NumPy — numerical arrays
- Pandas — tabular data
- Matplotlib / Seaborn — visualization
- SciPy — scientific statistics
- scikit-learn — preprocessing, PCA, splitting, and pipelines

The emphasis is on **why** each transformation is needed, not on memorizing API calls.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression

rng = np.random.default_rng(42)

print("Scientific Python environment ready.")


# 2. Create a synthetic materials dataset

We will create a dataset representing alloy/process experiments.

The dataset intentionally contains:

- numerical descriptors
- categorical variables
- missing values
- a few unusual observations
- correlated features
- redundant information

This allows us to reproduce realistic data-preparation problems.


In [ ]:
n = 300

alloy = rng.choice(
    ["A", "B", "C", "D"],
    size=n,
    p=[0.30, 0.25, 0.25, 0.20]
)

heat_treatment = rng.choice(
    ["As-quenched", "Tempered", "Annealed"],
    size=n,
    p=[0.35, 0.45, 0.20]
)

temperature_K = rng.choice(
    [773, 823, 873, 923, 973, 1023],
    size=n
)

time_h = rng.choice([1, 2, 4, 8, 16], size=n)

carbon_pct = np.clip(rng.normal(0.55, 0.12, n), 0.15, 0.95)
chromium_pct = np.clip(rng.normal(18.0, 1.2, n), 14, 22)
nickel_pct = np.clip(rng.normal(10.0, 1.0, n), 7, 13)
molybdenum_pct = np.clip(rng.normal(1.2, 0.5, n), 0, 3)

grain_size_um = (
    4.5
    + 0.005*(temperature_K - 773)
    + 0.5*np.log1p(time_h)
    + rng.normal(0, 0.7, n)
)

density_g_cm3 = 7.75 + rng.normal(0, 0.05, n)

porosity_pct = np.clip(
    1.2
    + 0.5*(heat_treatment == "Annealed")
    + rng.normal(0, 0.6, n),
    0.05,
    None
)

hardness_HV = (
    260
    + 35*carbon_pct
    + 10*chromium_pct
    + 18*molybdenum_pct
    - 7.5*grain_size_um
    - 12*porosity_pct
    + rng.normal(0, 9, n)
)

yield_strength_MPa = (
    100
    + 0.95*hardness_HV
    + 30*carbon_pct
    - 8*porosity_pct
    + rng.normal(0, 22, n)
)

tensile_strength_MPa = (
    yield_strength_MPa
    + 100
    + 12*chromium_pct
    - 10*porosity_pct
    + rng.normal(0, 28, n)
)

data = pd.DataFrame({
    "Sample_ID": [f"S{i:04d}" for i in range(1, n+1)],
    "Alloy": alloy,
    "Heat_Treatment": heat_treatment,
    "Temperature_K": temperature_K,
    "Time_h": time_h,
    "Carbon_pct": carbon_pct,
    "Chromium_pct": chromium_pct,
    "Nickel_pct": nickel_pct,
    "Molybdenum_pct": molybdenum_pct,
    "Grain_Size_um": grain_size_um,
    "Density_g_cm3": density_g_cm3,
    "Porosity_pct": porosity_pct,
    "Hardness_HV": hardness_HV,
    "Yield_Strength_MPa": yield_strength_MPa,
    "Tensile_Strength_MPa": tensile_strength_MPa
})

data.head()


# 3. Introduce realistic data-quality problems

Real materials datasets rarely arrive perfectly clean.

We will intentionally add:

- missing values
- duplicate rows
- extreme observations
- a few invalid values

The purpose is to practice a disciplined preprocessing workflow.


In [ ]:
dirty = data.copy()

# Missing values
for col, count in [
    ("Carbon_pct", 8),
    ("Grain_Size_um", 7),
    ("Hardness_HV", 6),
    ("Porosity_pct", 5)
]:
    idx = rng.choice(dirty.index, size=count, replace=False)
    dirty.loc[idx, col] = np.nan

# Duplicate a few rows
duplicates = dirty.sample(3, random_state=42)
dirty = pd.concat([dirty, duplicates], ignore_index=True)

# Inject a few unusual hardness values
outlier_idx = rng.choice(dirty.index, size=3, replace=False)
dirty.loc[outlier_idx, "Hardness_HV"] *= 1.7

# Inject an invalid negative porosity
invalid_idx = rng.choice(dirty.index, size=1, replace=False)
dirty.loc[invalid_idx, "Porosity_pct"] = -2

print("Shape:", dirty.shape)


# 4. Initial inspection

The first step is always to understand the data before transforming it.

Questions:

- What are the dimensions?
- Which variables are numerical?
- Which are categorical?
- How many missing values exist?
- Are there duplicates?
- Are units documented?


In [ ]:
print("Shape:", dirty.shape)

print("\nData types:")
print(dirty.dtypes)

print("\nMissing values:")
print(dirty.isna().sum())

print("\nDuplicate rows:")
print(dirty.duplicated().sum())


In [ ]:
dirty.describe(include="all").T


# Exercise 1 — Initial data audit

Create a short data-quality report containing:

1. number of rows
2. number of columns
3. categorical variables
4. numerical variables
5. missing values by variable
6. number of duplicate rows
7. minimum and maximum for every numerical variable

**Important:** Do not clean anything yet. First understand what is wrong.


# 5. Remove duplicate records

Duplicate observations can cause a model to effectively see the same sample more than once.

This is particularly dangerous when the same sample appears in both the training and test datasets.


In [ ]:
clean = dirty.drop_duplicates().copy()

print("Rows before:", len(dirty))
print("Rows after :", len(clean))


# 6. Validate physical ranges

Data cleaning should use **physical knowledge**, not only statistical rules.

Examples:

- porosity cannot be negative
- composition percentages should lie within meaningful ranges
- density should be positive
- grain size must be positive
- temperature in Kelvin should be positive

We will flag invalid porosity values.


In [ ]:
invalid_porosity = clean[clean["Porosity_pct"] < 0]

print("Invalid porosity observations:")
display(invalid_porosity)


In [ ]:
clean.loc[clean["Porosity_pct"] < 0, "Porosity_pct"] = np.nan


# 7. Missing-value strategies

Missing data can be handled by:

1. deleting observations
2. mean/median imputation
3. group-wise imputation
4. model-based imputation
5. algorithms that explicitly handle missing values

There is no universally correct method.

For skewed experimental variables, median imputation is often more robust than mean imputation.

However, in a real research project, the **reason for missingness** should be investigated first.


In [ ]:
missing_report = clean.isna().sum().sort_values(ascending=False)
missing_report


## Median imputation

For a variable \(x\), median imputation replaces a missing value with

\[
x_{\mathrm{missing}}\rightarrow \operatorname{median}(x_{\mathrm{observed}}).
\]

This is simple but reduces natural variability and should not be treated as measured data.


In [ ]:
imputed_demo = clean.copy()

for col in [
    "Carbon_pct",
    "Grain_Size_um",
    "Hardness_HV",
    "Porosity_pct"
]:
    imputed_demo[col] = imputed_demo[col].fillna(
        imputed_demo[col].median()
    )

print("Remaining missing values:",
      imputed_demo.isna().sum().sum())


# 8. Why preprocessing must avoid data leakage

Suppose we calculate the mean of a feature using the **entire dataset** and then split into train/test.

The test data have influenced the preprocessing parameters.

That is a form of **data leakage**.

Correct workflow:

\[
\boxed{
\text{split}
\rightarrow
\text{fit preprocessing on training data}
\rightarrow
\text{transform train and test}
}
\]

Incorrect workflow:

\[
\text{fit preprocessing on all data}
\rightarrow
\text{split}
\]

The difference becomes crucial when preprocessing is more complex than simple imputation.


# 9. Train-test split

Before fitting data-dependent transformations, create the split.

We will use 80% training data and 20% test data.

The target for later ML work will be tensile strength.


In [ ]:
target = "Tensile_Strength_MPa"

X = clean.drop(columns=[target, "Sample_ID"])
y = clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


## Stratification

For classification problems, stratification can preserve class proportions:

```python
train_test_split(
    X, y,
    test_size=0.2,
    stratify=y
)
```

For continuous regression targets, ordinary random splitting is usually appropriate, although specialized strategies can be useful for small or highly imbalanced datasets.


# 10. Separate numerical and categorical variables

Most ML algorithms require numerical arrays.

Our dataset contains both:

### Numerical
- temperature
- time
- composition
- grain size
- density
- porosity
- hardness
- yield strength

### Categorical
- alloy
- heat treatment

These need different preprocessing strategies.


In [ ]:
categorical_features = [
    "Alloy",
    "Heat_Treatment"
]

numerical_features = [
    c for c in X.columns
    if c not in categorical_features
]

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)


# 11. Scaling — mathematical foundation

Many ML methods depend on distances, dot products, or optimization.

Suppose a feature has mean \(\mu\) and standard deviation \(\sigma\).

Standardization gives:

\[
z=\frac{x-\mu}{\sigma}.
\]

After standardization, a feature has approximately:

\[
\mu_z=0,\qquad \sigma_z=1.
\]

This is especially important for:

- PCA
- k-nearest neighbors
- clustering
- support-vector machines
- many neural-network workflows
- regularized linear models


In [ ]:
scaler = StandardScaler()

demo = X_train[numerical_features].dropna()

scaled = scaler.fit_transform(demo)

scaled_df = pd.DataFrame(
    scaled,
    columns=numerical_features,
    index=demo.index
)

scaled_df.describe().T[["mean", "std"]].head()


# 12. Standardization versus normalization

These terms are often used loosely.

### Standardization

\[
z=\frac{x-\mu}{\sigma}
\]

centers and scales the feature.

### Min-max normalization

\[
x'=
\frac{x-x_{\min}}
{x_{\max}-x_{\min}}
\]

maps values approximately into \([0,1]\).

### Robust scaling

Uses median and interquartile range:

\[
x'=
\frac{x-\operatorname{median}(x)}
{\operatorname{IQR}(x)}.
\]

Robust scaling can be useful when strong outliers are present.


In [ ]:
standard = StandardScaler()
minmax = MinMaxScaler()
robust = RobustScaler()

example = X_train[["Hardness_HV"]].dropna()

comparison = pd.DataFrame({
    "Original": example.iloc[:, 0],
    "Standard": standard.fit_transform(example).ravel(),
    "MinMax": minmax.fit_transform(example).ravel(),
    "Robust": robust.fit_transform(example).ravel()
})

comparison.describe().T


# Exercise 2 — Scaling experiment

Choose two features with very different numerical ranges.

1. Plot them before scaling.
2. Apply standardization.
3. Compare their means and standard deviations.
4. Explain why scaling is important for PCA.
5. Explain why scaling may be unnecessary for a decision tree.


# 13. Encoding categorical variables

ML algorithms generally require numerical representations.

A categorical variable such as:

\[
\{\text{A},\text{B},\text{C},\text{D}\}
\]

can be represented using one-hot encoding:

\[
A\rightarrow(1,0,0,0)
\]

\[
B\rightarrow(0,1,0,0)
\]

and so on.

One-hot encoding does **not** imply that alloy D is numerically greater than alloy A.


In [ ]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded = encoder.fit_transform(
    X_train[categorical_features]
)

encoded_names = encoder.get_feature_names_out(
    categorical_features
)

encoded_df = pd.DataFrame(
    encoded,
    columns=encoded_names,
    index=X_train.index
)

encoded_df.head()


# 14. Why `handle_unknown="ignore"` matters

Suppose the training data contain alloys:

\[
A,B,C
\]

but the test set contains an unseen alloy:

\[
D.
\]

A robust preprocessing pipeline should not crash.

`handle_unknown="ignore"` allows unseen categories to be represented safely.

In real materials discovery, this issue becomes especially important when datasets from different sources contain different compositions or labels.


# 15. Feature engineering — physical descriptors

Feature engineering should be guided by physical knowledge.

Examples:

### Hall–Petch descriptor

\[
d^{-1/2}
\]

### Specific strength

\[
\frac{\sigma_y}{\rho}
\]

### Thermal exposure measure

A simple feature:

\[
T\log(1+t)
\]

### Composition-derived features

- total alloying content
- ratios
- atomic fractions
- valence electron concentration
- atomic-size mismatch

More advanced materials-informatics libraries such as **pymatgen** and **matminer** can automate chemically meaningful descriptor generation.


In [ ]:
feature_df = clean.copy()

feature_df["Inv_Sqrt_GrainSize"] = (
    1 / np.sqrt(feature_df["Grain_Size_um"])
)

feature_df["Specific_Yield_Strength"] = (
    feature_df["Yield_Strength_MPa"]
    / feature_df["Density_g_cm3"]
)

feature_df["Thermal_Exposure"] = (
    feature_df["Temperature_K"]
    * np.log1p(feature_df["Time_h"])
)

feature_df[
    [
        "Grain_Size_um",
        "Inv_Sqrt_GrainSize",
        "Yield_Strength_MPa",
        "Density_g_cm3",
        "Specific_Yield_Strength",
        "Thermal_Exposure"
    ]
].head()


# 16. Outlier detection with the IQR rule

For a feature \(x\):

\[
IQR=Q_3-Q_1
\]

and potential outliers may satisfy

\[
x<Q_1-1.5IQR
\]

or

\[
x>Q_3+1.5IQR.
\]

This is a **flagging method**, not an automatic deletion rule.


In [ ]:
def iqr_outlier_mask(series, factor=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor*iqr
    upper = q3 + factor*iqr
    return (series < lower) | (series > upper)

mask = iqr_outlier_mask(clean["Hardness_HV"])

print("Potential hardness outliers:", mask.sum())

display(
    clean.loc[
        mask,
        ["Sample_ID", "Alloy", "Hardness_HV", "Grain_Size_um"]
    ]
)


# 17. Z-score outlier detection

The z-score is

\[
z_i=\frac{x_i-\mu}{\sigma}.
\]

A common heuristic is to flag observations with

\[
|z|>3.
\]

This method assumes that the mean and standard deviation are meaningful summaries.

It can therefore be sensitive to extreme values.


In [ ]:
hardness = clean["Hardness_HV"].dropna()

z = np.abs(stats.zscore(hardness))

print("Potential |z| > 3 outliers:", (z > 3).sum())


# 18. Compare IQR and z-score approaches

There is no universally superior outlier detector.

### IQR
- robust to extreme values
- useful for skewed distributions
- easy to explain

### z-score
- natural for approximately Gaussian distributions
- sensitive to mean/std distortion

### Domain rules
Often the most important approach.

For example, a measured grain size of 0.001 μm might be statistically unusual but physically impossible for the experimental system.


# 19. Feature selection

A dataset may contain:

- irrelevant features
- redundant features
- noisy features
- highly correlated features
- features that are unavailable at prediction time

Feature selection can improve:

- interpretability
- computational efficiency
- generalization
- model stability

We will first examine correlation with the target.


In [ ]:
feature_target_corr = (
    clean[numerical_features + [target]]
    .corr()[target]
    .drop(target)
    .sort_values(key=np.abs, ascending=False)
)

feature_target_corr


## Warning: correlation is not feature selection

A low Pearson correlation does not necessarily mean a feature is useless.

A nonlinear relationship may have weak linear correlation.

Similarly, a high correlation may arise because two variables measure nearly the same physical quantity.


# 20. Univariate feature selection

`SelectKBest` can score numerical features against a target.

For regression, `f_regression` evaluates linear dependence.

This is useful as an educational example, but feature selection should be performed **inside cross-validation** for rigorous model evaluation.


In [ ]:
selection_df = clean[
    numerical_features + [target]
].dropna()

X_num = selection_df[numerical_features]
y_num = selection_df[target]

selector = SelectKBest(
    score_func=f_regression,
    k=5
)

X_selected = selector.fit_transform(X_num, y_num)

selected_features = X_num.columns[
    selector.get_support()
]

print("Selected features:")
print(selected_features.tolist())


# 21. Correlated and redundant features

Suppose:

\[
x_2\approx 3x_1.
\]

Both may carry nearly the same information.

Highly correlated features can cause:

- redundancy
- unstable regression coefficients
- difficulty interpreting models
- unnecessary dimensionality

However, correlation should not be removed blindly. Two strongly correlated variables may represent different physical mechanisms.


In [ ]:
corr = clean[numerical_features].corr()

plt.figure(figsize=(11,8))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0
)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()


# 22. PCA — Principal Component Analysis

PCA transforms correlated features into orthogonal directions.

Given a centered/scaled data matrix \(X\), PCA seeks directions \(w_k\) maximizing projected variance:

\[
w_k=
\arg\max_{\|w\|=1}
\operatorname{Var}(Xw).
\]

The principal components satisfy:

\[
w_i^Tw_j=0,\qquad i\neq j.
\]

PCA is closely related to eigenvalue problems and SVD, linking this module directly to the linear algebra of Module 2.


## Important: scale before PCA

If one variable has values around \(10^4\) and another around \(10^{-2}\), the large-scale variable can dominate the variance.

For many materials datasets, standardization is therefore an essential PCA preprocessing step.


In [ ]:
pca_features = [
    "Temperature_K",
    "Time_h",
    "Carbon_pct",
    "Chromium_pct",
    "Nickel_pct",
    "Molybdenum_pct",
    "Grain_Size_um",
    "Density_g_cm3",
    "Porosity_pct",
    "Hardness_HV",
    "Yield_Strength_MPa"
]

pca_data = clean[pca_features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_data)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print("Original dimensions:", X_scaled.shape[1])
print("PCA dimensions:", X_pca.shape[1])


# 23. Explained variance

Each principal component has an associated explained-variance ratio:

\[
r_k=
\frac{\lambda_k}
{\sum_j\lambda_j}.
\]

The cumulative explained variance tells us how many components are needed to retain a chosen fraction of the total variance.


In [ ]:
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

for i, (single, cum) in enumerate(
    zip(explained, cumulative), start=1
):
    print(
        f"PC{i:2d}: "
        f"{single:.3f} individual, "
        f"{cum:.3f} cumulative"
    )


In [ ]:
plt.figure(figsize=(7,4))
components = np.arange(1, len(explained)+1)

plt.plot(
    components,
    cumulative,
    marker="o"
)

plt.axhline(
    0.90,
    linestyle="--",
    label="90%"
)

plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA explained variance")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 24. PCA scores

Plot the first two principal components.

Each point represents one material/sample.

Distances in this standardized PCA space indicate similarity with respect to the selected features.


In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=clean.loc[pca_data.index, "Alloy"],
    style=clean.loc[pca_data.index, "Heat_Treatment"],
    s=70
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Materials represented in PCA space")
plt.grid(alpha=0.25)
plt.show()


# 25. PCA loadings

The PCA components are vectors.

The loading of feature \(j\) on component \(k\) is related to:

\[
w_{kj}.
\]

Large absolute loadings indicate features strongly contributing to that component.

This can provide physical interpretation.


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=pca_features,
    columns=[
        f"PC{i}"
        for i in range(1, len(pca_features)+1)
    ]
)

loadings[["PC1", "PC2", "PC3"]].sort_values(
    "PC1",
    key=np.abs,
    ascending=False
)


### Interpretation exercise

For PC1 and PC2:

1. identify the three largest absolute loadings
2. determine whether they are positive or negative
3. suggest a physical interpretation
4. explain what information is lost by reducing the data to two dimensions


# 26. PCA reconstruction

PCA can also be viewed as a compression method.

If only the first \(k\) components are retained:

\[
X\approx Z_kW_k.
\]

Let's reconstruct the standardized data using different numbers of components.


In [ ]:
for k in [2, 3, 5, 8]:
    pca_k = PCA(n_components=k)
    Z = pca_k.fit_transform(X_scaled)
    X_reconstructed = pca_k.inverse_transform(Z)

    reconstruction_error = np.mean(
        (X_scaled - X_reconstructed)**2
    )

    print(
        f"k={k:2d} | "
        f"explained={pca_k.explained_variance_ratio_.sum():.3f} | "
        f"MSE={reconstruction_error:.5f}"
    )


# 27. Avoiding PCA leakage

PCA learns directions from the data.

Therefore, PCA fitted using both training and test data leaks information from the test set.

Correct:

```text
split
  ↓
fit scaler on training
  ↓
transform training
  ↓
fit PCA on transformed training
  ↓
transform test using training PCA
```

This principle is one reason scikit-learn pipelines are so valuable.


# 28. Build a preprocessing pipeline

We now construct a reusable pipeline.

### Numerical branch

\[
\text{impute median}
\rightarrow
\text{standardize}
\]

### Categorical branch

\[
\text{impute most frequent}
\rightarrow
\text{one-hot encode}
\]

Then the two branches are combined.


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


# 29. Inspect transformed feature names

A preprocessing pipeline can produce a feature matrix with more columns than the original dataset because categorical variables are one-hot encoded.


In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

print("\nFirst 20:")
print(feature_names[:20])


# 30. The complete preprocessing architecture

We have now implemented:

\[
\boxed{
X
\rightarrow
\begin{cases}
\text{Numerical}\rightarrow
\text{Imputation}\rightarrow
\text{Scaling}\\
\text{Categorical}\rightarrow
\text{Imputation}\rightarrow
\text{One-hot encoding}
\end{cases}
\rightarrow
X_{\mathrm{ML}}
}
\]

This transformed matrix is the type of input expected by many machine-learning algorithms.


# 31. Pipeline + PCA

We can extend the numerical branch with PCA.

A simplified numerical workflow becomes:

\[
\text{imputation}
\rightarrow
\text{scaling}
\rightarrow
\text{PCA}.
\]

In a real ML workflow, the number of components should be selected using the training data and, where appropriate, cross-validation.


In [ ]:
numeric_pca_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90))
])

X_train_num = X_train[numerical_features]
X_test_num = X_test[numerical_features]

X_train_pca = numeric_pca_pipeline.fit_transform(X_train_num)
X_test_pca = numeric_pca_pipeline.transform(X_test_num)

pca_model = numeric_pca_pipeline.named_steps["pca"]

print("Original numerical features:", len(numerical_features))
print("PCA components retained:", pca_model.n_components_)
print(
    "Explained variance:",
    pca_model.explained_variance_ratio_.sum()
)


# 32. Data leakage demonstration

Consider standardization.

### Wrong

```python
scaler.fit(pd.concat([X_train, X_test]))
```

### Correct

```python
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

The second approach ensures the test set remains genuinely unseen.

This distinction is essential when reporting ML performance in research papers.


# 33. Feature engineering challenge

Create at least five new features using physical reasoning.

Possible ideas:

### Grain-size physics

\[
d^{-1/2}
\]

### Specific strength

\[
\sigma_y/\rho
\]

### Thermal exposure

\[
T\log(1+t)
\]

### Total alloying content

\[
C+Cr+Ni+Mo
\]

### Composition ratio

\[
Cr/Ni
\]

Do not create features merely because they are mathematically possible. Explain the physical motivation for every engineered descriptor.


In [ ]:
challenge = clean.copy()

challenge["Inv_Sqrt_GrainSize"] = (
    1 / np.sqrt(challenge["Grain_Size_um"])
)

challenge["Specific_Yield_Strength"] = (
    challenge["Yield_Strength_MPa"]
    / challenge["Density_g_cm3"]
)

challenge["Thermal_Exposure"] = (
    challenge["Temperature_K"]
    * np.log1p(challenge["Time_h"])
)

challenge["Total_Alloying"] = (
    challenge["Carbon_pct"]
    + challenge["Chromium_pct"]
    + challenge["Nickel_pct"]
    + challenge["Molybdenum_pct"]
)

challenge["Cr_to_Ni"] = (
    challenge["Chromium_pct"]
    / challenge["Nickel_pct"]
)

challenge[
    [
        "Inv_Sqrt_GrainSize",
        "Specific_Yield_Strength",
        "Thermal_Exposure",
        "Total_Alloying",
        "Cr_to_Ni"
    ]
].head()


# 34. Integrated materials-data preparation workflow

Let's formulate the complete workflow as an algorithm.

### Input

Raw materials dataset \(D\).

### Step 1

Remove exact duplicates.

### Step 2

Validate physical ranges.

### Step 3

Define target \(y\) and features \(X\).

### Step 4

Split into training/test data.

### Step 5

Separate numerical/categorical variables.

### Step 6

Fit preprocessing on training data only.

### Step 7

Apply transformations to training and test data.

### Step 8

Engineer scientifically justified descriptors.

### Step 9

Perform feature selection or dimensionality reduction.

### Step 10

Export the processed data/model pipeline for ML.

This workflow becomes the foundation for Modules 6+.


# 35. Hands-on Exercise A — Data cleaning

Using `dirty`:

1. Remove duplicates.
2. Identify physically invalid values.
3. Replace invalid values with missing values.
4. Quantify missingness.
5. Identify potential outliers.
6. Produce a final cleaned dataset.
7. Document every decision.

**Deliverable:** a short data-cleaning report.


# 36. Hands-on Exercise B — Scaling

Select:

- grain size
- density
- hardness
- yield strength

Compare:

1. raw distributions
2. standardized distributions
3. min-max scaled distributions
4. robust-scaled distributions

Answer:

> Which scaling method would you choose for this dataset and why?


# 37. Hands-on Exercise C — Feature engineering

Create at least five materials descriptors.

For every descriptor provide:

- mathematical definition
- Python implementation
- physical motivation
- units
- expected effect on a relevant property

Avoid arbitrary polynomial features unless you can justify them.


# 38. Hands-on Exercise D — PCA

Perform PCA on the numerical descriptors.

1. Standardize the data.
2. Calculate PCA.
3. Plot explained variance.
4. Select the minimum number of components explaining at least 90% variance.
5. Plot PC1 vs PC2.
6. Color by alloy.
7. Interpret the largest loadings.
8. Discuss information loss.


# 39. Hands-on Exercise E — Detecting leakage

Create two workflows:

### Workflow 1 — intentionally wrong

Fit a scaler on the complete dataset before splitting.

### Workflow 2 — correct

Split first and fit the scaler only on training data.

Explain why the second workflow is scientifically valid and the first is not.

Do **not** use the leaked workflow to report model performance.


# 40. Mini-project — Materials Informatics Data Preparation

## Problem statement

You are given a raw materials dataset containing composition, processing parameters, microstructure descriptors, and mechanical properties.

Your task is to produce a **machine-learning-ready dataset**.

### Required components

#### 1. Data audit
- dimensions
- data types
- missing values
- duplicates
- physical validity

#### 2. Data cleaning
Document every transformation.

#### 3. Outlier analysis
Use at least two methods and compare the results.

#### 4. Feature engineering
Create at least five physically motivated descriptors.

#### 5. Categorical encoding
Encode all categorical variables appropriately.

#### 6. Scaling
Compare at least two scaling methods.

#### 7. Feature selection
Use correlation analysis and at least one formal feature-selection method.

#### 8. PCA
Perform PCA and explain the dominant components.

#### 9. Train/test preparation
Demonstrate that all learned preprocessing parameters come from training data only.

#### 10. Final deliverable
Produce:

- cleaned dataset
- preprocessing notebook
- feature dictionary
- PCA plots
- data-quality report
- short scientific interpretation


# 41. Recommended grading rubric

| Component | Marks |
|---|---:|
| Data audit | 10 |
| Cleaning and physical validation | 15 |
| Missing/outlier handling | 15 |
| Feature engineering | 15 |
| Encoding and scaling | 10 |
| Feature selection | 10 |
| PCA and interpretation | 15 |
| Leakage-free workflow | 5 |
| Scientific documentation | 5 |
| **Total** | **100** |


# 42. Connection to machine learning

At the end of Module 5, the data are ready for supervised learning.

The progression is:

\[
\boxed{
\text{Raw materials data}
\rightarrow
\text{clean data}
\rightarrow
\text{descriptors}
\rightarrow
\text{feature matrix}
\rightarrow
\text{ML model}
}
\]

For example:

\[
X=
\begin{bmatrix}
C & Cr & Ni & Mo & d & \rho & P & T & t & \cdots
\end{bmatrix}
\]

and the target could be:

\[
y=\sigma_y.
\]

The next question becomes:

\[
\boxed{
\text{Can we learn }f(X)\approx y?
}
\]

That is the transition from **data preparation** to **machine learning**.


# 43. Connection to computational materials

For computational materials science, the same workflow applies to:

### DFT data

\[
\text{structure}
\rightarrow
\text{composition/descriptors}
\rightarrow
E,\;E_g,\;V,\;\text{magnetic properties}
\]

### Molecular dynamics

\[
\text{trajectory}
\rightarrow
\text{statistical descriptors}
\rightarrow
\text{transport/mechanical properties}
\]

### Phase-field simulations

\[
\text{field evolution}
\rightarrow
\text{microstructure descriptors}
\rightarrow
\text{property prediction}
\]

### Experimental materials databases

\[
\text{composition + processing + microstructure}
\rightarrow
\text{properties}
\]

In every case, the representation of the material is central.


# 44. Recommended materials-informatics libraries

The core stack introduced here can later be extended with:

### `pymatgen`
Crystal structures, compositions, phase diagrams, transformations, and materials-science analysis.

### `matminer`
Materials descriptors and featurization workflows.

### `scikit-learn`
Preprocessing, PCA, feature selection, classical ML, model validation.

### `SciPy`
Numerical algorithms and scientific statistics.

### `ase`
Atomic simulation environments and structure/trajectory manipulation.

### `h5py`
Efficient storage and retrieval of large numerical datasets.

### `pathlib`
Portable file and directory management.

These tools become particularly important when students move from synthetic classroom datasets to DFT, MD, experimental, or materials-database workflows.


# 45. Final checklist

Before giving a dataset to a machine-learning algorithm, ask:

### Data
- What does each row represent?
- What does each column represent?
- Are the units known?

### Quality
- Are there duplicates?
- Are values physically plausible?
- Are there missing values?
- Are there outliers?

### Representation
- Are categorical variables encoded?
- Are descriptors physically meaningful?
- Are redundant variables present?

### Scaling
- Does the algorithm require scaling?
- Was scaling fitted only on training data?

### Leakage
- Did any test information influence preprocessing?
- Are any features derived from the target?
- Is any future information included?

### Dimensionality
- Is PCA appropriate?
- How much variance is retained?
- Can the components be physically interpreted?

### Reproducibility
- Can another researcher reproduce the preprocessing?
- Are all decisions documented?
- Is the pipeline saved?

If these questions cannot be answered, the dataset is **not yet ready for machine learning**.


# 46. Key takeaways

The most important concepts from Module 5 are:

\[
\boxed{\text{Clean before learning}}
\]

\[
\boxed{\text{Use physics to engineer descriptors}}
\]

\[
\boxed{\text{Fit preprocessing on training data only}}
\]

\[
\boxed{\text{Scaling matters for distance/variance-based methods}}
\]

\[
\boxed{\text{PCA connects data science to linear algebra}}
\]

\[
\boxed{\text{Feature engineering is a scientific modeling decision}}
\]

The ultimate objective is not to create the largest possible feature matrix.

It is to create a **physically meaningful, statistically sound, leakage-free representation of materials** that can support reliable machine learning.
